![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Custom Data with Ticker Mapping Research

This notebook loads the mapped trade data from the Object Store and shows how the FB ticker resolves to the current META symbol.

### Set Up QuantBook

Save the selected-trades file to the Object Store for the custom data reader.

In [ ]:
SELECTED_TRADES_FILE = "selected_trades.csv"

CONTENT = """
2020-01-20,FB,100
2020-01-20,MSFT,200
2020-01-20,NVDA,300
2024-09-03,META,-100
2024-09-03,MSFT,-200
2024-09-03,NVDA,-300"""

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 12, 31)
qb.object_store.save(SELECTED_TRADES_FILE, CONTENT)

### Define the Custom Data Type

Define a [custom securities](https://www.quantconnect.com/docs/v2/writing-algorithms/importing-data/streaming-data/custom-securities) reader that maps each point-in-time ticker to a Symbol.

In [ ]:
class SelectedTrades(PythonData):

    def get_source(self, config: SubscriptionDataConfig, date: datetime, is_live_mode: bool) -> SubscriptionDataSource:
        return SubscriptionDataSource(SELECTED_TRADES_FILE, SubscriptionTransportMedium.OBJECT_STORE)

    def reader(self, config: SubscriptionDataConfig, line: str, date: datetime, is_live_mode: bool) -> BaseData:
        if not line.strip():
            return None
        data = [x.strip() for x in line.split(',')]
        ticker = data[1]
        time = datetime.strptime(data[0], "%Y-%m-%d")
        # Resolve the point-in-time SecurityIdentifier so FB maps to today's META symbol.
        security_id = SecurityIdentifier.generate_equity(ticker, Market.USA, mapping_resolve_date=time)
        if security_id.date.year < 1998:
            return None
        trade = SelectedTrades()
        trade.symbol = Symbol(security_id, ticker)
        trade.end_time = time
        trade.time = time - timedelta(1)
        trade.quantity = float(data[2])
        return trade

In [ ]:
trades = qb.add_data(SelectedTrades, "X")

### Build Time Series

Request the full history of the mapped trade data and inspect the resolved symbols and quantities.

In [ ]:
history = qb.history(SelectedTrades, trades.symbol, datetime(2020, 1, 1), datetime(2024, 12, 31))
history

In [ ]:
# Extract the unique asset Symbols from the custom data history.
asset_symbols = list(history.index.levels[0])
# Request daily history for the assets referenced by the custom data.
asset_history = qb.history(asset_symbols, datetime(2020, 1, 1), datetime(2024, 12, 31), Resolution.DAILY)
asset_history